In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("../data/processed/cleaned_online_retail.csv", parse_dates=["InvoiceDate"])

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalAmount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [ ]:
#Find the Latest Transaction Date

In [3]:
snapshot_date = df["InvoiceDate"].max()

print(snapshot_date)

2011-12-09 12:50:00


In [ ]:
#Create RFM Features

In [ ]:
#Recency

In [4]:
recency = (
    snapshot_date
    - df.groupby("Customer ID")["InvoiceDate"].max()
).dt.days

In [ ]:
#Frequency

In [5]:
frequency = (
    df.groupby("Customer ID")["Invoice"]
      .nunique()
)

In [ ]:
#Monetary

In [6]:
monetary = (
    df.groupby("Customer ID")["TotalAmount"]
      .sum()
)

In [ ]:
#Combine RFM

In [7]:
rfm = pd.concat(
    [recency, frequency, monetary],
    axis=1
)

rfm.columns = [
    "Recency",
    "Frequency",
    "Monetary"
]

rfm.head()

,Recency,Frequency,Monetary
Customer ID,,,
12346,325,1,77183.60
12347,1,7,4310.00
12348,74,4,1797.24
12349,18,1,1757.55
12350,309,1,334.40


In [ ]:
#Average Order Value

In [8]:
average_order = (
    df.groupby("Customer ID")["TotalAmount"]
      .mean()
)

rfm["AverageOrderValue"] = average_order

In [ ]:
#Total Quantity Purchased

In [9]:
rfm["TotalQuantity"] = (
    df.groupby("Customer ID")["Quantity"]
      .sum()
)

In [ ]:
#Number of Unique Products

In [10]:
rfm["UniqueProducts"] = (
    df.groupby("Customer ID")["StockCode"]
      .nunique()
)

In [ ]:
#Customer Lifetime

In [11]:
first_purchase = (
    df.groupby("Customer ID")["InvoiceDate"]
      .min()
)

In [12]:
last_purchase = (
    df.groupby("Customer ID")["InvoiceDate"]
      .max()
)

In [13]:
#Lifetime
rfm["CustomerLifetime"] = (
    last_purchase - first_purchase
).dt.days

In [ ]:
#Average Days Between Purchases

In [14]:
purchase_gap = (
    df.sort_values("InvoiceDate")
      .groupby("Customer ID")["InvoiceDate"]
      .diff()
      .dt.days
)

avg_gap = (
    purchase_gap.groupby(df["Customer ID"])
               .mean()
)

rfm["AveragePurchaseGap"] = avg_gap

In [ ]:
#Fill missing values

In [15]:
rfm["AveragePurchaseGap"] = (
    rfm["AveragePurchaseGap"]
       .fillna(0)
)

In [ ]:
#Define Churn
If a customer hasn't purchased in the last 90 days, they are considered churned.

In [16]:
rfm["Churn"] = np.where(
    rfm["Recency"] > 90,
    1,
    0
)

In [ ]:
#Check Class Balance

In [17]:
rfm["Churn"].value_counts()

Churn
0    2893
1    1445
Name: count, dtype: int64

In [18]:
rfm["Churn"].value_counts(normalize=True) * 100

Churn
0    66.689719
1    33.310281
Name: proportion, dtype: float64

In [ ]:
#Final Dataset

In [19]:
rfm.head()

,Recency,Frequency,Monetary,AverageOrderValue,TotalQuantity,UniqueProducts,CustomerLifetime,AveragePurchaseGap,Churn
Customer ID,,,,,,,,,
12346,325,1,77183.60,77183.600000,74215,1,0,0.0,1
12347,1,7,4310.00,23.681319,2458,103,365,2.0,0
12348,74,4,1797.24,57.975484,2341,22,282,9.4,0
12349,18,1,1757.55,24.076027,631,73,0,0.0,0
12350,309,1,334.40,19.670588,197,17,0,0.0,1


In [ ]:
rfm.to_csv("../data/processed/customer_features.csv")